In [10]:
import pandas as pd
import numpy as np
from modulos.data_load import load, coins
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit

In [ ]:

resultados = {}
n_splits = 5

for coin in coins:
    df = load(coin)
    #print(f"{coin} →", df.columns.tolist())
    print(f'Processando moeda: {coin}')

    df = df.sort_values('date').reset_index(drop=True)
    
    # Features derivadas
    ## Preço
    df['faixa_preco'] = df['high'] - df['low']
    df['retorno_pct_7d'] = df['close'].pct_change(7)
    df['momentum_7d'] = df['close'] - df['close'].shift(7)

    ## Indicadores estatísticos
    df['media_movel_7d'] = df['close'].rolling(window=7).mean()
    df['std_7d'] = df['close'].rolling(window=7).std()

    ## Liquidez
    df['volume_2_7d'] = df['Volume 2'].rolling(window=7).mean()
    df['taker_ratio'] = df['buyTakerAmount'] / df['Volume 2'].replace(0, np.nan)
    df['buy_pressure'] = df['buyTakerQuantity'] / df['tradeCount'].replace(0, np.nan)
    df['volume_volatilidade_ratio'] = df['Volume 2'] / df['faixa_preco'].replace(0, np.nan)


    df['date'] = pd.to_datetime(df['date'])
    df['dia_da_semana'] = df['date'].dt.dayofweek
    
    # Limpando NaNs
    df.dropna(inplace=True)
    
    # Separa os conjuntos
    n = len(df)
    n_train = int(n * 0.7)
    n_val = int(n * 0.15)

    df_train = df.iloc[:n_train]
    df_val = df.iloc[n_train:n_train + n_val]
    df_test = df.iloc[n_train + n_val:]

    # Normaliza
    features = [
        'media_movel_7d',
        'std_7d', 'momentum_7d', 'retorno_pct_7d',
        'volume_2_7d', 'taker_ratio', 'buy_pressure', 'volume_volatilidade_ratio',
        'dia_da_semana'
    ]
    
    # TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(df)):
        df_train = df.iloc[train_idx].copy()
        df_val   = df.iloc[val_idx].copy()

        # Normaliza por fold
        scaler = MinMaxScaler()
        scaler.fit(df_train[features])

        df_train.loc[:, features] = scaler.transform(df_train[features])
        df_val.loc[:, features]   = scaler.transform(df_val[features])

        #print(f"{coin} — Fold {fold_idx+1}: Train {df_train.shape}, Val {df_val.shape}")

        folds.append({
            'train': df_train,
            'val':   df_val,
            'scaler': scaler
        })
    resultados[coin] = folds
print(resultados)

Processando moeda: AAVEBTC
Processando moeda: AAVEUSDT


/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.5        0.66666667 ... 0.         0.         0.        ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_train.loc[:, features] = scaler.transform(df_train[features])
/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_val.loc[:, features]   = scaler.transform(df_val[features])
/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. 

Processando moeda: ACMUSDD
Processando moeda: ADAUSDT


/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.         0.         0.         0.         0.
 0.         0.16666667 0.16666667 0.16666667 0.16666667 0.16666667
 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667
 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667
 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667 0.16666667
 0.16666667 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333
 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333
 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333
 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333 0.33333333
 0.33333333 0.5        0.5        0.5        0.5        0.5
 0.5        0.5        0.5        0.5        0.5        0.5
 0.5        0.5        0.5        0.5        0.5        0.5
 0.5        0.5        0

Processando moeda: BNBUSDT
Processando moeda: BNTUSDT
Processando moeda: CVTBTC


...

Processando moeda: DOGEBTC
Processando moeda: ETCETH


/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1.         0.         0.16666667 ... 0.         0.         0.        ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_train.loc[:, features] = scaler.transform(df_train[features])
/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.         0.         ... 0.83333333 1.         1.        ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_val.loc[:, features]   = scaler.transform(df_val[features])
/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:63: FutureWarning: Setting an item of incompatible dtype is depreca

Processando moeda: USDPUSDT


/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1.         0.5        0.5        0.5        0.66666667 1.
 1.         0.         0.         0.         0.         0.16666667
 0.16666667 0.16666667 0.16666667 0.16666667]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_train.loc[:, features] = scaler.transform(df_train[features])
/var/folders/z7/g5fntl4s2q31td292ttsm5x80000gn/T/ipykernel_13351/1735909447.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.16666667 0.16666667 0.33333333 0.5        0.5        1.
 1.         0.33333333 0.33333333 0.5        1.         1.
 0.33333333 0.16666667]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_val.loc[:, features]   = scaler.tran

{'AAVEBTC': [{'train':                 unix                date    symbol      open      high  \
15823  1659312000000 2022-08-01 00:00:00  AAVE/BTC  0.004295  0.004420   
15908  1659618000000 2022-08-04 13:00:00  AAVE/BTC  0.004195  0.004196   
15927  1659686400000 2022-08-05 08:00:00  AAVE/BTC  0.002978  0.004235   
15928  1659690000000 2022-08-05 09:00:00  AAVE/BTC  0.004099  0.004246   
15929  1659693600000 2022-08-05 10:00:00  AAVE/BTC  0.004346  0.004632   
...              ...                 ...       ...       ...       ...   
18522  1669028400000 2022-11-21 11:00:00  AAVE/BTC  0.003509  0.003511   
18523  1669032000000 2022-11-21 12:00:00  AAVE/BTC  0.003491  0.003500   
18524  1669035600000 2022-11-21 13:00:00  AAVE/BTC  0.003495  0.003510   
18525  1669039200000 2022-11-21 14:00:00  AAVE/BTC  0.003491  0.003526   
18526  1669042800000 2022-11-21 15:00:00  AAVE/BTC  0.003511  0.003524   

            low     close  Volume 1    Volume 2  buyTakerAmount  ...  \
15823  0.004196 